# Week 3 — Model Comparison & Forecast Export

**Goal:** Put the moving-average baseline and the ARIMA model side by side, score
both with MAPE and RMSE, pick the winner per category, and export the final
forecasts plus a scoreboard for the Week 4 dashboard.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

PROC = Path('..') / 'data' / 'processed'
DAILY_PATH = PROC / 'supply_chain_daily.csv'
ARIMA_PATH = PROC / 'arima_forecast.csv'
FORECAST_OUT = PROC / 'forecasts.csv'
SCORES_OUT = PROC / 'model_scores.csv'
TARGET = 'units_sold'
HORIZON = 30
WINDOW = 7
SAMPLE_CAT = 'Electronics'

daily = pd.read_csv(DAILY_PATH, parse_dates=['date'])
daily = daily.sort_values(['product_category', 'date']).reset_index(drop=True)
print('Loaded daily shape:', daily.shape)

## 1. Rebuild the baseline and load ARIMA

The baseline is cheap to recompute here; the ARIMA forecasts are read from the
CSV saved by `arima_forecast.ipynb` (run that notebook first).

In [ ]:
def train_test_split(frame, horizon=HORIZON):
    train_parts, test_parts = [], []
    for cat, sub in frame.groupby('product_category'):
        sub = sub.sort_values('date')
        train_parts.append(sub.iloc[:-horizon])
        test_parts.append(sub.iloc[-horizon:])
    return pd.concat(train_parts, ignore_index=True), pd.concat(test_parts, ignore_index=True)

def moving_average_forecast(train_frame, test_frame, window=WINDOW):
    parts = []
    for cat, sub in test_frame.groupby('product_category'):
        sub = sub.sort_values('date').copy()
        last_window = (train_frame[train_frame['product_category'] == cat]
                       .sort_values('date')[TARGET].tail(window))
        sub['forecast_ma'] = round(last_window.mean(), 2)
        parts.append(sub[['date', 'product_category', TARGET, 'forecast_ma']])
    return pd.concat(parts, ignore_index=True)

train, test = train_test_split(daily)
baseline = moving_average_forecast(train, test)
arima = pd.read_csv(ARIMA_PATH, parse_dates=['date'])
print('Baseline rows:', len(baseline), '| ARIMA rows:', len(arima))

## 2. Merge and score both models

Join the two forecasts on date + category, then compute MAPE and RMSE per model
per category.

In [ ]:
def mape(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    mask = actual != 0
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100)

def rmse(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return float(np.sqrt(np.mean((actual - predicted) ** 2)))

forecasts = baseline.merge(
    arima[['date', 'product_category', 'forecast_arima']],
    on=['date', 'product_category'], how='inner')
print('Merged forecast rows:', len(forecasts))

rows = []
for cat, sub in forecasts.groupby('product_category'):
    rows.append({
        'product_category': cat,
        'MAPE_baseline': round(mape(sub[TARGET], sub['forecast_ma']), 2),
        'MAPE_arima': round(mape(sub[TARGET], sub['forecast_arima']), 2),
        'RMSE_baseline': round(rmse(sub[TARGET], sub['forecast_ma']), 2),
        'RMSE_arima': round(rmse(sub[TARGET], sub['forecast_arima']), 2),
    })
model_scores = pd.DataFrame(rows)
model_scores['winner'] = np.where(
    model_scores['RMSE_arima'] <= model_scores['RMSE_baseline'], 'ARIMA', 'baseline')
model_scores = model_scores.sort_values('product_category').reset_index(drop=True)
model_scores

## 3. Overall winner and export

Compare the two models across all categories, then save the merged forecasts and
the scoreboard for the dashboard.

In [ ]:
overall_baseline_rmse = rmse(forecasts[TARGET], forecasts['forecast_ma'])
overall_arima_rmse = rmse(forecasts[TARGET], forecasts['forecast_arima'])
overall_winner = 'ARIMA' if overall_arima_rmse <= overall_baseline_rmse else 'baseline'

print('Overall RMSE  baseline:', round(overall_baseline_rmse, 2))
print('Overall RMSE  ARIMA   :', round(overall_arima_rmse, 2))
print('Overall winner        :', overall_winner)
print('Per-category wins:')
print(model_scores['winner'].value_counts())

forecasts.to_csv(FORECAST_OUT, index=False)
model_scores.to_csv(SCORES_OUT, index=False)
print('Saved', FORECAST_OUT)
print('Saved', SCORES_OUT)

## 4. Visualize both models vs actual

Overlay the actual test demand with both forecasts for one category to see which
model tracks reality better.

In [ ]:
one = forecasts[forecasts['product_category'] == SAMPLE_CAT].sort_values('date')

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(one['date'], one[TARGET], color='black', lw=1.8, label='actual (test)')
ax.plot(one['date'], one['forecast_ma'], color='red', lw=1.4, ls='--', label='baseline (moving avg)')
ax.plot(one['date'], one['forecast_arima'], color='crimson', lw=1.4, ls=':', label='ARIMA')
ax.set_title(f'{SAMPLE_CAT}: baseline vs ARIMA vs actual')
ax.set_xlabel('date'); ax.set_ylabel(TARGET); ax.legend()
plt.tight_layout()
plt.show()